In [1]:
# import kagglehub

# # Download latest version into the "data" folder
# path = kagglehub.dataset_download(
#     "nickfratto/pacs-dataset",
#     output_dir="data"
# )

# print("Path to dataset files:", path)

In [2]:
import os
import numpy as np
import torch

from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split


# ============================================================
# Configuration
# ============================================================

DATA_ROOT = "../workspace/data/pacs_data/pacs_data"

SOURCE_DOMAINS = [
    "photo",
    "art_painting",
    "cartoon",
]

TARGET_DOMAIN = "sketch"

SEED = 6304
BATCH_SIZE = 32
NUM_WORKERS = 4


# ============================================================
# Reproducibility
# ============================================================

torch.manual_seed(SEED)
np.random.seed(SEED)

generator = torch.Generator()
generator.manual_seed(SEED)


# ============================================================
# Transform
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# Create stratified train/validation split
# ============================================================

def create_source_split(domain):

    domain_path = os.path.join(DATA_ROOT, domain)

    dataset = datasets.ImageFolder(
        domain_path,
        transform=transform
    )

    labels = np.array(dataset.targets)
    indices = np.arange(len(dataset))

    train_idx, val_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=SEED,
        stratify=labels
    )

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    return train_subset, val_subset


# ============================================================
# Split each source domain independently
# ============================================================

source_train_sets = []
source_val_sets = []

for domain in SOURCE_DOMAINS:

    train_set, val_set = create_source_split(domain)

    source_train_sets.append(train_set)
    source_val_sets.append(val_set)

    print(
        f"{domain:15s} | "
        f"train = {len(train_set):5d} | "
        f"val = {len(val_set):5d}"
    )


# ============================================================
# Combine source domains
# ============================================================

source_train_dataset = ConcatDataset(source_train_sets)
source_val_dataset = ConcatDataset(source_val_sets)


# ============================================================
# Target domain: Sketch
# ============================================================

target_path = os.path.join(DATA_ROOT, TARGET_DOMAIN)

target_dataset = datasets.ImageFolder(
    target_path,
    transform=transform
)


# ============================================================
# DataLoaders
# ============================================================

train_loader_source = DataLoader(
    source_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=generator
)

val_loader_source = DataLoader(
    source_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

target_loader = DataLoader(
    target_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


# ============================================================
# Sanity checks
# ============================================================

print("\nDataset sizes")
print("-" * 45)
print(f"Source train : {len(source_train_dataset)}")
print(f"Source val   : {len(source_val_dataset)}")
print(f"Target       : {len(target_dataset)}")

print("\nTarget classes:")
print(target_dataset.classes)

print("\nClass -> index:")
print(target_dataset.class_to_idx)

photo           | train =  1336 | val =   334
art_painting    | train =  1638 | val =   410
cartoon         | train =  1875 | val =   469

Dataset sizes
---------------------------------------------
Source train : 4849
Source val   : 1213
Target       : 3929

Target classes:
['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']

Class -> index:
{'dog': 0, 'elephant': 1, 'giraffe': 2, 'guitar': 3, 'horse': 4, 'house': 5, 'person': 6}


# ERM Baseline

In [3]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights


class PACSResNet18(nn.Module):
    """
    ResNet-18 for PACS.

    Architecture:
        Image
          ↓
        ResNet-18 backbone
          ↓
        512-dimensional feature vector
          ↓
        Linear classifier
          ↓
        7-class logits

    The 512-D features can later be used for DAN/MMD alignment.
    """

    def __init__(self, num_classes=7):
        super().__init__()

        # ----------------------------------------------------
        # Pretrained ResNet-18
        # ----------------------------------------------------
        weights = ResNet18_Weights.IMAGENET1K_V1
        backbone = resnet18(weights=weights)

        # Everything except the original ImageNet classifier.
        #
        # Output after avgpool:
        #     [batch_size, 512, 1, 1]
        self.feature_extractor = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        # ----------------------------------------------------
        # PACS classifier
        # ----------------------------------------------------
        self.classifier = nn.Linear(
            in_features=512,
            out_features=num_classes
        )


    def forward(self, x, return_features=False):

        # [B, 3, 224, 224]
        features = self.feature_extractor(x)

        # [B, 512, 1, 1] -> [B, 512]
        features = torch.flatten(features, 1)

        # [B, 512] -> [B, 7]
        logits = self.classifier(features)

        if return_features:
            return logits, features

        return logits

In [4]:
SEED = 6304

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Initialize model architecture
model = PACSResNet18(num_classes=7).to(device)

# Load trained parameters
state_dict = torch.load(
    "erm-baseline.pt",
    map_location=device,
    weights_only=True
)

model.load_state_dict(state_dict)

# Evaluation mode
model.eval()

print("ERM baseline loaded successfully.")

ERM baseline loaded successfully.


In [5]:
from sklearn.metrics import accuracy_score, f1_score


# ============================================================
# ERM Baseline — Source-Domain Validation
# ============================================================

def evaluate_model(model, loader, device):
    """
    Evaluate a model and return accuracy and macro-F1.
    """
    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            predictions = logits.argmax(dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return accuracy, macro_f1


# ============================================================
# Build one validation loader per source domain
# ============================================================

source_val_loaders = {
    domain: DataLoader(
        val_set,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    for domain, val_set in zip(SOURCE_DOMAINS, source_val_sets)
}


# ============================================================
# Evaluate ERM checkpoint on each source domain separately
# ============================================================

erm_results = {}

for domain in SOURCE_DOMAINS:
    accuracy, macro_f1 = evaluate_model(
        model,
        source_val_loaders[domain],
        device
    )

    erm_results[domain] = {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }


# ============================================================
# Mean and worst-domain performance
# ============================================================

accuracies = [
    erm_results[d]["accuracy"]
    for d in SOURCE_DOMAINS
]

macro_f1s = [
    erm_results[d]["macro_f1"]
    for d in SOURCE_DOMAINS
]

mean_accuracy = np.mean(accuracies)
mean_macro_f1 = np.mean(macro_f1s)

# "Worst" means the lowest-performing source domain.
worst_accuracy = np.min(accuracies)
worst_macro_f1 = np.min(macro_f1s)

worst_accuracy_domain = SOURCE_DOMAINS[np.argmin(accuracies)]
worst_f1_domain = SOURCE_DOMAINS[np.argmin(macro_f1s)]


# ============================================================
# Report results
# ============================================================

display_names = {
    "photo": "Photo",
    "art_painting": "Art Painting",
    "cartoon": "Cartoon"
}

print("\nERM Baseline — Source Validation Results")
print("=" * 65)
print(f"{'Domain':<18} {'Accuracy':>15} {'Macro-F1':>15}")
print("-" * 65)

for domain in SOURCE_DOMAINS:
    print(
        f"{display_names[domain]:<18} "
        f"{erm_results[domain]['accuracy']:>15.4f} "
        f"{erm_results[domain]['macro_f1']:>15.4f}"
    )

print("-" * 65)
print(
    f"{'Mean':<18} "
    f"{mean_accuracy:>15.4f} "
    f"{mean_macro_f1:>15.4f}"
)
print(
    f"{'Worst-domain':<18} "
    f"{worst_accuracy:>15.4f} "
    f"{worst_macro_f1:>15.4f}"
)
print("=" * 65)

print(
    f"\nLowest accuracy : "
    f"{display_names[worst_accuracy_domain]} "
    f"({worst_accuracy:.4f})"
)

print(
    f"Lowest macro-F1 : "
    f"{display_names[worst_f1_domain]} "
    f"({worst_macro_f1:.4f})"
)


ERM Baseline — Source Validation Results
Domain                    Accuracy        Macro-F1
-----------------------------------------------------------------
Photo                       0.9671          0.9599
Art Painting                0.9146          0.9168
Cartoon                     0.9446          0.9455
-----------------------------------------------------------------
Mean                        0.9421          0.9407
Worst-domain                0.9146          0.9168

Lowest accuracy : Art Painting (0.9146)
Lowest macro-F1 : Art Painting (0.9168)


# DAN-DG – Pairwise Source-Domain Alignment

In [6]:
# ------------------------------------------------------------
# MMD with three RBF kernels
# ------------------------------------------------------------
def mmd_loss(source, target):
    z = torch.cat([source, target], dim=0)

    # Pairwise squared feature distances
    d2 = torch.cdist(z, z).pow(2)

    # Median pairwise squared distance (excluding diagonal)
    with torch.no_grad():
        mask = torch.triu(
            torch.ones_like(d2, dtype=torch.bool),
            diagonal=1
        )
        median_d2 = d2[mask].median().clamp_min(1e-8)

    bandwidths = [
        0.5 * median_d2,
        1.0 * median_d2,
        2.0 * median_d2
    ]

    K = sum(torch.exp(-d2 / bw) for bw in bandwidths)

    ns = source.size(0)

    K_ss = K[:ns, :ns]
    K_tt = K[ns:, ns:]
    K_st = K[:ns, ns:]

    return K_ss.mean() + K_tt.mean() - 2 * K_st.mean()



In [7]:
import copy
from itertools import cycle
from sklearn.metrics import f1_score

# ------------------------------------------------------------
# DAN-DG — Pairwise Source-Domain Alignment
# Fresh model — DO NOT load the ERM baseline
# ------------------------------------------------------------

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

dan_dg_model = PACSResNet18(num_classes=7).to(device)

# 8 samples from each source domain.
# Keep domains separate because MMD is computed pairwise.
source_loaders = [
    DataLoader(
        ds,
        batch_size=8,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    for ds in source_train_sets
]

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    dan_dg_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

LAMBDA_DG = 1.0


# ------------------------------------------------------------
# MMD^2 with three RBF kernels
#
# For each source-domain pair, bandwidths are:
#   0.5, 1.0, 2.0 x median pairwise squared distance
# in the current batch.
# ------------------------------------------------------------

def mmd_loss(source, target):
    """
    Biased empirical MMD^2 between two feature batches.

    source: [Ns, 512]
    target: [Nt, 512]
    """

    z = torch.cat([source, target], dim=0)

    # Pairwise squared Euclidean distances
    d2 = torch.cdist(z, z).pow(2)

    # Median pairwise squared distance, excluding diagonal
    with torch.no_grad():
        mask = torch.triu(
            torch.ones_like(d2, dtype=torch.bool),
            diagonal=1
        )

        median_d2 = d2[mask].median().clamp_min(1e-8)

    bandwidths = [
        0.5 * median_d2,
        1.0 * median_d2,
        2.0 * median_d2
    ]

    # Sum of the three RBF kernels
    K = sum(
        torch.exp(-d2 / bw)
        for bw in bandwidths
    )

    ns = source.size(0)

    K_ss = K[:ns, :ns]
    K_tt = K[ns:, ns:]
    K_st = K[:ns, ns:]

    # MMD^2
    return (
        K_ss.mean()
        + K_tt.mean()
        - 2.0 * K_st.mean()
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

def evaluate_dan_dg(loader):
    dan_dg_model.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = dan_dg_model(x)

            preds.extend(
                logits.argmax(dim=1).cpu().numpy()
            )
            labels.extend(y.numpy())

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    acc = np.mean(preds == labels)
    f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    return acc, f1


# ------------------------------------------------------------
# Train DAN-DG
#
# Objective:
#
# L_DAN-DG =
#     L_ERM
#     + (lambda_DG / 3) *
#       [MMD^2(F(X0), F(X1))
#        + MMD^2(F(X0), F(X2))
#        + MMD^2(F(X1), F(X2))]
#
# IMPORTANT:
# No target/Sketch samples are used during training.
# ------------------------------------------------------------

best_f1 = -1.0
best_state = None
bad_epochs = 0

for epoch in range(30):

    dan_dg_model.train()

    # Cycle each source domain independently
    source_iters = [
        cycle(loader)
        for loader in source_loaders
    ]

    # One epoch lasts as long as the largest source loader
    steps = max(
        len(loader)
        for loader in source_loaders
    )

    epoch_cls_loss = 0.0
    epoch_mmd_loss = 0.0
    epoch_total_loss = 0.0

    for _ in range(steps):

        domain_x = []
        domain_y = []

        # ----------------------------------------------------
        # Get 8 examples independently from each source domain
        # ----------------------------------------------------

        for src_iter in source_iters:
            x, y = next(src_iter)

            domain_x.append(x.to(device))
            domain_y.append(y.to(device))

        optimizer.zero_grad()

        domain_logits = []
        domain_features = []

        # ----------------------------------------------------
        # Forward each domain separately so its 512-D features
        # remain identifiable for pairwise MMD.
        # ----------------------------------------------------

        for x in domain_x:

            logits, features = dan_dg_model(
                x,
                return_features=True
            )

            domain_logits.append(logits)
            domain_features.append(features)

        # ----------------------------------------------------
        # ERM classification loss over ALL source labels
        # ----------------------------------------------------

        all_logits = torch.cat(
            domain_logits,
            dim=0
        )

        all_labels = torch.cat(
            domain_y,
            dim=0
        )

        cls_loss = criterion(
            all_logits,
            all_labels
        )

        # ----------------------------------------------------
        # Three unordered source-domain pairs
        #
        # For three source domains:
        #   (0, 1)
        #   (0, 2)
        #   (1, 2)
        # ----------------------------------------------------

        mmd_01 = mmd_loss(
            domain_features[0],
            domain_features[1]
        )

        mmd_02 = mmd_loss(
            domain_features[0],
            domain_features[2]
        )

        mmd_12 = mmd_loss(
            domain_features[1],
            domain_features[2]
        )

        # Average MMD discrepancy over the three pairs
        pairwise_mmd = (
            mmd_01
            + mmd_02
            + mmd_12
        ) / 3.0

        # lambda_DG = 1
        loss = (
            cls_loss
            + LAMBDA_DG * pairwise_mmd
        )

        loss.backward()
        optimizer.step()

        epoch_cls_loss += cls_loss.item()
        epoch_mmd_loss += pairwise_mmd.item()
        epoch_total_loss += loss.item()

    # --------------------------------------------------------
    # Epoch averages
    # --------------------------------------------------------

    epoch_cls_loss /= steps
    epoch_mmd_loss /= steps
    epoch_total_loss /= steps

    # --------------------------------------------------------
    # Source-validation performance
    # --------------------------------------------------------

    val_acc, val_f1 = evaluate_dan_dg(
        val_loader_source
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f} | "
        f"Cls Loss: {epoch_cls_loss:.4f} | "
        f"Pairwise MMD: {epoch_mmd_loss:.4f} | "
        f"Total Loss: {epoch_total_loss:.4f}"
    )

    # --------------------------------------------------------
    # Early stopping using source-validation macro-F1
    # --------------------------------------------------------

    if val_f1 > best_f1:

        best_f1 = val_f1

        best_state = copy.deepcopy(
            dan_dg_model.state_dict()
        )

        bad_epochs = 0

    else:
        bad_epochs += 1

    if bad_epochs >= 5:
        print("Early stopping.")
        break


# ------------------------------------------------------------
# Restore and save best DAN-DG checkpoint
# ------------------------------------------------------------

dan_dg_model.load_state_dict(best_state)

torch.save(
    dan_dg_model.state_dict(),
    "dan-dg-pairwise-mmd.pt"
)

print(
    f"Saved checkpoint: dan-dg-pairwise-mmd.pt | "
    f"Best source-val Macro-F1: {best_f1:.4f}"
)

Epoch 01 | Val Acc: 0.9209 | Val Macro-F1: 0.9208 | Cls Loss: 0.4598 | Pairwise MMD: 0.2371 | Total Loss: 0.6968
Epoch 02 | Val Acc: 0.9349 | Val Macro-F1: 0.9353 | Cls Loss: 0.1888 | Pairwise MMD: 0.2301 | Total Loss: 0.4189
Epoch 03 | Val Acc: 0.9151 | Val Macro-F1: 0.9148 | Cls Loss: 0.1803 | Pairwise MMD: 0.2253 | Total Loss: 0.4057
Epoch 04 | Val Acc: 0.9382 | Val Macro-F1: 0.9398 | Cls Loss: 0.1903 | Pairwise MMD: 0.2299 | Total Loss: 0.4202
Epoch 05 | Val Acc: 0.9291 | Val Macro-F1: 0.9292 | Cls Loss: 0.2008 | Pairwise MMD: 0.2358 | Total Loss: 0.4365
Epoch 06 | Val Acc: 0.9464 | Val Macro-F1: 0.9467 | Cls Loss: 0.1881 | Pairwise MMD: 0.2215 | Total Loss: 0.4096
Epoch 07 | Val Acc: 0.9382 | Val Macro-F1: 0.9374 | Cls Loss: 0.2008 | Pairwise MMD: 0.2316 | Total Loss: 0.4324
Epoch 08 | Val Acc: 0.9340 | Val Macro-F1: 0.9349 | Cls Loss: 0.1924 | Pairwise MMD: 0.2287 | Total Loss: 0.4211
Epoch 09 | Val Acc: 0.9373 | Val Macro-F1: 0.9377 | Cls Loss: 0.1985 | Pairwise MMD: 0.2323 | To

In [8]:
# ============================================================
# DAN-DG — Final Results
# ============================================================

from sklearn.metrics import accuracy_score, f1_score
import numpy as np


def evaluate_dan_dg(model, loader):
    model.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = model(x)
            pred = logits.argmax(dim=1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.numpy())

    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    return accuracy, macro_f1


# Load best DAN-DG model
dan_dg_model.load_state_dict(
    torch.load(
        "dan-dg-pairwise-mmd.pt",
        map_location=device,
        weights_only=True
    )
)

dan_dg_model.eval()


# Source validation results
source_acc, source_f1 = evaluate_dan_dg(
    dan_dg_model,
    val_loader_source
)

# Target (Sketch) results
target_acc, target_f1 = evaluate_dan_dg(
    dan_dg_model,
    target_loader
)


# Display results
print("\nDAN-DG Results")
print("=" * 50)

print(
    f"Source Validation | "
    f"Accuracy: {source_acc:.4f} | "
    f"Macro-F1: {source_f1:.4f}"
)

print(
    f"Target (Sketch)   | "
    f"Accuracy: {target_acc:.4f} | "
    f"Macro-F1: {target_f1:.4f}"
)

print("=" * 50)


DAN-DG Results
Source Validation | Accuracy: 0.9464 | Macro-F1: 0.9467
Target (Sketch)   | Accuracy: 0.5846 | Macro-F1: 0.5281


# SAM – Parameter-Space Stability

In [9]:
# ============================================================
# SAM — Parameter-Space Stability
#
# Objective:
#     min_theta max_{||epsilon||_2 <= rho} L_ERM(theta + epsilon)
#
# Standard, non-adaptive SAM:
#   1. Compute gradient at theta
#   2. Perturb parameters in normalized gradient direction
#   3. Compute loss/gradient at theta + epsilon
#   4. Restore theta and update it using the perturbed gradient
#
# IMPORTANT:
#   - rho = 0.05
#   - AdamW with same LR and weight decay as ERM
#   - BatchNorm running statistics are frozen during BOTH passes
#   - Only source-domain data are used for training
# ============================================================

import copy
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Fresh model
#
# SAM is trained as its own method; do not start from the
# already-trained ERM checkpoint.
# ------------------------------------------------------------

sam_model = PACSResNet18(
    num_classes=7
).to(device)


# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------

RHO = 0.05
SAM_EPS = 1e-12
NUM_EPOCHS = 30

criterion = nn.CrossEntropyLoss()

# Same optimizer settings used by ERM / the notebook:
# AdamW, lr=1e-4, weight_decay=1e-4.
optimizer_sam = torch.optim.AdamW(
    sam_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# Frozen BatchNorm running-statistics policy
#
# model.train() is still used so the rest of the network is in
# training mode, but BatchNorm layers are switched to eval mode.
#
# Therefore:
#   - running_mean / running_var are NOT updated
#   - learned affine parameters gamma/beta still receive gradients
#
# This function is called before BOTH SAM passes.
# ------------------------------------------------------------

def freeze_batchnorm_running_stats(model):
    for module in model.modules():
        if isinstance(
            module,
            nn.modules.batchnorm._BatchNorm
        ):
            module.eval()


# ------------------------------------------------------------
# Gradient norm used by standard non-adaptive SAM
#
# ||g||_2 = sqrt(sum_i ||grad_i||_2^2)
# ------------------------------------------------------------

def sam_gradient_norm(model):

    norms = []

    for parameter in model.parameters():

        if parameter.grad is not None:
            norms.append(
                parameter.grad.detach().norm(p=2)
            )

    if len(norms) == 0:
        return torch.tensor(
            0.0,
            device=device
        )

    return torch.norm(
        torch.stack(norms),
        p=2
    )


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

def evaluate_sam(model, loader):

    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(images)

            predictions = logits.argmax(
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return accuracy, macro_f1


# ============================================================
# Train SAM
# ============================================================

best_f1 = -1.0
best_state = None


for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Training mode, followed by frozen BatchNorm statistics.
    # --------------------------------------------------------

    sam_model.train()
    freeze_batchnorm_running_stats(sam_model)

    epoch_first_loss = 0.0
    epoch_sam_loss = 0.0

    num_batches = 0


    for images, labels in train_loader_source:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        # ====================================================
        # SAM PASS 1
        #
        # Compute:
        #     g = grad L(theta)
        #
        # Then construct:
        #     epsilon =
        #       rho * g / (||g||_2 + eps)
        # ====================================================

        sam_model.train()
        freeze_batchnorm_running_stats(sam_model)

        optimizer_sam.zero_grad()

        logits = sam_model(images)

        first_loss = criterion(
            logits,
            labels
        )

        first_loss.backward()


        # ----------------------------------------------------
        # Global L2 norm of the gradient
        # ----------------------------------------------------

        grad_norm = sam_gradient_norm(
            sam_model
        )

        scale = (
            RHO
            / (grad_norm + SAM_EPS)
        )


        # ----------------------------------------------------
        # Move from theta to theta + epsilon.
        #
        # Save each perturbation so the original parameters
        # can be restored exactly after the second pass.
        # ----------------------------------------------------

        perturbations = {}

        with torch.no_grad():

            for parameter in sam_model.parameters():

                if parameter.grad is None:
                    continue

                epsilon = (
                    parameter.grad
                    * scale
                )

                parameter.add_(epsilon)

                perturbations[parameter] = epsilon


        # ====================================================
        # SAM PASS 2
        #
        # Evaluate:
        #     L(theta + epsilon)
        #
        # and obtain its gradient.
        #
        # BatchNorm running statistics remain frozen here too.
        # ====================================================

        optimizer_sam.zero_grad()

        sam_model.train()
        freeze_batchnorm_running_stats(sam_model)

        perturbed_logits = sam_model(
            images
        )

        sam_loss = criterion(
            perturbed_logits,
            labels
        )

        sam_loss.backward()


        # ----------------------------------------------------
        # Restore the ORIGINAL parameters:
        #
        #     theta + epsilon -> theta
        #
        # Gradients are intentionally retained because they are
        # the gradients of L(theta + epsilon), which AdamW must
        # use to update the original theta.
        # ----------------------------------------------------

        with torch.no_grad():

            for parameter, epsilon in perturbations.items():
                parameter.sub_(epsilon)


        # ----------------------------------------------------
        # Update ORIGINAL parameters using gradient evaluated
        # at the perturbed point.
        # ----------------------------------------------------

        optimizer_sam.step()


        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        epoch_first_loss += first_loss.item()
        epoch_sam_loss += sam_loss.item()

        num_batches += 1


    # --------------------------------------------------------
    # Epoch averages
    # --------------------------------------------------------

    epoch_first_loss /= num_batches
    epoch_sam_loss /= num_batches


    # --------------------------------------------------------
    # Source-domain validation
    #
    # Model selection uses source validation only.
    # Sketch is NOT used for training/model selection.
    # --------------------------------------------------------

    val_acc, val_f1 = evaluate_sam(
        sam_model,
        val_loader_source
    )


    print(
        f"Epoch {epoch + 1:02d} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f} | "
        f"ERM Loss: {epoch_first_loss:.4f} | "
        f"SAM Loss: {epoch_sam_loss:.4f}"
    )


    # --------------------------------------------------------
    # Keep checkpoint with best source-validation Macro-F1
    # --------------------------------------------------------

    if val_f1 > best_f1:

        best_f1 = val_f1

        best_state = copy.deepcopy(
            sam_model.state_dict()
        )


# ============================================================
# Restore and save best SAM checkpoint
# ============================================================

sam_model.load_state_dict(
    best_state
)

torch.save(
    sam_model.state_dict(),
    "sam-rho-0.05.pt"
)

print(
    "\nSaved checkpoint: sam-rho-0.05.pt | "
    f"Best source-val Macro-F1: {best_f1:.4f}"
)


# ============================================================
# Final SAM evaluation
# ============================================================

sam_model.eval()


# ------------------------------------------------------------
# Individual source-domain validation results
# ------------------------------------------------------------

sam_results = {}

for domain in SOURCE_DOMAINS:

    accuracy, macro_f1 = evaluate_sam(
        sam_model,
        source_val_loaders[domain]
    )

    sam_results[domain] = {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }


# ------------------------------------------------------------
# Mean / worst source-domain performance
# ------------------------------------------------------------

source_accuracies = [
    sam_results[d]["accuracy"]
    for d in SOURCE_DOMAINS
]

source_macro_f1s = [
    sam_results[d]["macro_f1"]
    for d in SOURCE_DOMAINS
]

mean_source_accuracy = np.mean(
    source_accuracies
)

mean_source_f1 = np.mean(
    source_macro_f1s
)

worst_source_accuracy = np.min(
    source_accuracies
)

worst_source_f1 = np.min(
    source_macro_f1s
)


# ------------------------------------------------------------
# Unseen target domain: Sketch
# ------------------------------------------------------------

target_accuracy, target_macro_f1 = evaluate_sam(
    sam_model,
    target_loader
)


# ============================================================
# Report
# ============================================================

display_names = {
    "photo": "Photo",
    "art_painting": "Art Painting",
    "cartoon": "Cartoon"
}


print("\nSAM (rho = 0.05) — Final Results")
print("=" * 70)

print(
    f"{'Domain':<20}"
    f"{'Accuracy':>15}"
    f"{'Macro-F1':>15}"
)

print("-" * 70)


for domain in SOURCE_DOMAINS:

    print(
        f"{display_names[domain]:<20}"
        f"{sam_results[domain]['accuracy']:>15.4f}"
        f"{sam_results[domain]['macro_f1']:>15.4f}"
    )


print("-" * 70)

print(
    f"{'Source Mean':<20}"
    f"{mean_source_accuracy:>15.4f}"
    f"{mean_source_f1:>15.4f}"
)

print(
    f"{'Source Worst':<20}"
    f"{worst_source_accuracy:>15.4f}"
    f"{worst_source_f1:>15.4f}"
)

print("-" * 70)

print(
    f"{'Target (Sketch)':<20}"
    f"{target_accuracy:>15.4f}"
    f"{target_macro_f1:>15.4f}"
)

print("=" * 70)

Epoch 01 | Val Acc: 0.8904 | Val Macro-F1: 0.8921 | ERM Loss: 0.7851 | SAM Loss: 1.0925
Epoch 02 | Val Acc: 0.9299 | Val Macro-F1: 0.9297 | ERM Loss: 0.2285 | SAM Loss: 0.4661
Epoch 03 | Val Acc: 0.9349 | Val Macro-F1: 0.9370 | ERM Loss: 0.1231 | SAM Loss: 0.3060
Epoch 04 | Val Acc: 0.9382 | Val Macro-F1: 0.9404 | ERM Loss: 0.0613 | SAM Loss: 0.1955
Epoch 05 | Val Acc: 0.9489 | Val Macro-F1: 0.9500 | ERM Loss: 0.0369 | SAM Loss: 0.1586
Epoch 06 | Val Acc: 0.9398 | Val Macro-F1: 0.9403 | ERM Loss: 0.0177 | SAM Loss: 0.1105
Epoch 07 | Val Acc: 0.9472 | Val Macro-F1: 0.9481 | ERM Loss: 0.0108 | SAM Loss: 0.0877
Epoch 08 | Val Acc: 0.9505 | Val Macro-F1: 0.9510 | ERM Loss: 0.0056 | SAM Loss: 0.0727
Epoch 09 | Val Acc: 0.9184 | Val Macro-F1: 0.9206 | ERM Loss: 0.0061 | SAM Loss: 0.0752
Epoch 10 | Val Acc: 0.9423 | Val Macro-F1: 0.9439 | ERM Loss: 0.0066 | SAM Loss: 0.0819
Epoch 11 | Val Acc: 0.9464 | Val Macro-F1: 0.9494 | ERM Loss: 0.0064 | SAM Loss: 0.0749
Epoch 12 | Val Acc: 0.9481 | Val

# Common Evaluation and Diagnostics

In [10]:
# ============================================================
# Common Evaluation and Diagnostics
#
# For ERM, DAN-DG, and SAM report:
#   1. Accuracy and macro-F1 on each source validation domain
#   2. Mean source-domain accuracy and macro-F1
#   3. Worst source-domain accuracy and macro-F1
#   4. Final accuracy and macro-F1 on Sketch
#   5. Change in Sketch accuracy relative to ERM
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# Common evaluation function
# ------------------------------------------------------------

def evaluate_common(model, loader):
    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            logits = model(images)

            predictions = logits.argmax(dim=1)

            all_labels.extend(
                labels.numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return accuracy, macro_f1


# ------------------------------------------------------------
# Make sure each method uses its saved best checkpoint
# ------------------------------------------------------------

# ERM
model.load_state_dict(
    torch.load(
        "erm-baseline.pt",
        map_location=device,
        weights_only=True
    )
)
model.eval()


# DAN-DG
dan_dg_model.load_state_dict(
    torch.load(
        "dan-dg-pairwise-mmd.pt",
        map_location=device,
        weights_only=True
    )
)
dan_dg_model.eval()


# SAM
sam_model.load_state_dict(
    torch.load(
        "sam-rho-0.05.pt",
        map_location=device,
        weights_only=True
    )
)
sam_model.eval()


# ------------------------------------------------------------
# Models to compare
# ------------------------------------------------------------

models = {
    "ERM": model,
    "DAN-DG": dan_dg_model,
    "SAM": sam_model
}


display_names = {
    "photo": "Photo",
    "art_painting": "Art Painting",
    "cartoon": "Cartoon"
}


# ------------------------------------------------------------
# Evaluate every method using exactly the same procedure
# ------------------------------------------------------------

common_results = {}

for method_name, current_model in models.items():

    method_results = {}

    # --------------------------------------------------------
    # Individual source validation domains
    # --------------------------------------------------------

    for domain in SOURCE_DOMAINS:

        acc, f1 = evaluate_common(
            current_model,
            source_val_loaders[domain]
        )

        method_results[domain] = {
            "accuracy": acc,
            "macro_f1": f1
        }


    # --------------------------------------------------------
    # Mean and worst-domain source performance
    # --------------------------------------------------------

    source_accs = [
        method_results[d]["accuracy"]
        for d in SOURCE_DOMAINS
    ]

    source_f1s = [
        method_results[d]["macro_f1"]
        for d in SOURCE_DOMAINS
    ]

    method_results["mean_accuracy"] = np.mean(
        source_accs
    )

    method_results["mean_macro_f1"] = np.mean(
        source_f1s
    )

    method_results["worst_accuracy"] = np.min(
        source_accs
    )

    method_results["worst_macro_f1"] = np.min(
        source_f1s
    )

    method_results["worst_accuracy_domain"] = (
        SOURCE_DOMAINS[np.argmin(source_accs)]
    )

    method_results["worst_f1_domain"] = (
        SOURCE_DOMAINS[np.argmin(source_f1s)]
    )


    # --------------------------------------------------------
    # Unseen target domain: Sketch
    # --------------------------------------------------------

    sketch_acc, sketch_f1 = evaluate_common(
        current_model,
        target_loader
    )

    method_results["sketch_accuracy"] = sketch_acc
    method_results["sketch_macro_f1"] = sketch_f1


    common_results[method_name] = method_results


# ============================================================
# Detailed report for each method
# ============================================================

for method_name in models:

    r = common_results[method_name]

    print(
        f"\n{method_name} — Common Evaluation"
    )
    print("=" * 72)

    print(
        f"{'Domain':<22}"
        f"{'Accuracy':>15}"
        f"{'Macro-F1':>15}"
    )

    print("-" * 72)

    for domain in SOURCE_DOMAINS:

        print(
            f"{display_names[domain]:<22}"
            f"{r[domain]['accuracy']:>15.4f}"
            f"{r[domain]['macro_f1']:>15.4f}"
        )

    print("-" * 72)

    print(
        f"{'Source Mean':<22}"
        f"{r['mean_accuracy']:>15.4f}"
        f"{r['mean_macro_f1']:>15.4f}"
    )

    print(
        f"{'Source Worst':<22}"
        f"{r['worst_accuracy']:>15.4f}"
        f"{r['worst_macro_f1']:>15.4f}"
    )

    print("-" * 72)

    print(
        f"{'Sketch':<22}"
        f"{r['sketch_accuracy']:>15.4f}"
        f"{r['sketch_macro_f1']:>15.4f}"
    )

    print("=" * 72)

    print(
        "Worst accuracy domain : "
        f"{display_names[r['worst_accuracy_domain']]}"
    )

    print(
        "Worst macro-F1 domain : "
        f"{display_names[r['worst_f1_domain']]}"
    )


# ============================================================
# Compact comparison table
# ============================================================

summary_rows = []

for method_name in models:

    r = common_results[method_name]

    row = {
        "Method": method_name
    }

    for domain in SOURCE_DOMAINS:

        name = display_names[domain]

        row[f"{name} Acc"] = (
            r[domain]["accuracy"]
        )

        row[f"{name} F1"] = (
            r[domain]["macro_f1"]
        )

    row["Mean Acc"] = r["mean_accuracy"]
    row["Mean F1"] = r["mean_macro_f1"]

    row["Worst Acc"] = r["worst_accuracy"]
    row["Worst F1"] = r["worst_macro_f1"]

    row["Sketch Acc"] = r["sketch_accuracy"]
    row["Sketch F1"] = r["sketch_macro_f1"]

    summary_rows.append(row)


summary_df = pd.DataFrame(
    summary_rows
).set_index("Method")


print("\n\nOverall Comparison")
print("=" * 100)

display(
    summary_df.round(4)
)


# ============================================================
# Change in Sketch accuracy relative to ERM
# ============================================================

erm_sketch_acc = common_results[
    "ERM"
]["sketch_accuracy"]


print("\nSketch Accuracy Change Relative to ERM")
print("=" * 55)

for method_name in ["DAN-DG", "SAM"]:

    method_acc = common_results[
        method_name
    ]["sketch_accuracy"]

    # Absolute change in accuracy, expressed as percentage points.
    delta = method_acc - erm_sketch_acc

    print(
        f"{method_name:<10}: "
        f"{delta:+.4f} "
        f"({delta * 100:+.2f} percentage points)"
    )

print("=" * 55)


ERM — Common Evaluation
Domain                       Accuracy       Macro-F1
------------------------------------------------------------------------
Photo                          0.9671         0.9599
Art Painting                   0.9146         0.9168
Cartoon                        0.9446         0.9455
------------------------------------------------------------------------
Source Mean                    0.9421         0.9407
Source Worst                   0.9146         0.9168
------------------------------------------------------------------------
Sketch                         0.6103         0.5948
Worst accuracy domain : Art Painting
Worst macro-F1 domain : Art Painting

DAN-DG — Common Evaluation
Domain                       Accuracy       Macro-F1
------------------------------------------------------------------------
Photo                          0.9850         0.9819
Art Painting                   0.9220         0.9199
Cartoon                        0.9403         0.944

,Photo Acc,Photo F1,Art Painting Acc,Art Painting F1,Cartoon Acc,Cartoon F1,Mean Acc,Mean F1,Worst Acc,Worst F1,Sketch Acc,Sketch F1
Method,,,,,,,,,,,,
ERM,0.9671,0.9599,0.9146,0.9168,0.9446,0.9455,0.9421,0.9407,0.9146,0.9168,0.6103,0.5948
DAN-DG,0.9850,0.9819,0.9220,0.9199,0.9403,0.9441,0.9491,0.9486,0.9220,0.9199,0.5846,0.5281
SAM,0.9790,0.9748,0.9390,0.9421,0.9638,0.9690,0.9606,0.9620,0.9390,0.9421,0.7157,0.7295



Sketch Accuracy Change Relative to ERM
DAN-DG    : -0.0257 (-2.57 percentage points)
SAM       : +0.1054 (+10.54 percentage points)


## Adding a Normalized Perturbation

In [11]:
# ============================================================
# Local Sharpness Diagnostic
#
# For each model:
#
#   Delta_sharp = L_val(theta + epsilon) - L_val(theta)
#
#   epsilon = 0.05 * grad(L_val) / ||grad(L_val)||_2
#
# Protocol:
#   - Same fixed validation batch for ALL models
#   - 32 examples from EACH source domain (96 total)
#   - Sampling seed = 6304
#   - Models are kept in evaluation mode
#   - Cross-entropy loss
#   - One normalized gradient-ascent perturbation
#   - Radius rho = 0.05
#   - Original parameters are restored exactly afterward
# ============================================================

import torch
import torch.nn as nn
import pandas as pd


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SHARPNESS_SEED = 6304
SHARPNESS_RHO = 0.05
EXAMPLES_PER_SOURCE = 32

sharpness_criterion = nn.CrossEntropyLoss()


# ============================================================
# 1. Construct ONE fixed validation batch
#    containing exactly 32 examples from each source domain.
#
# The SAME batch is reused for ERM, DAN-DG, and SAM.
# ============================================================

sharpness_generator = torch.Generator()
sharpness_generator.manual_seed(SHARPNESS_SEED)

batch_images = []
batch_labels = []

for domain, val_set in zip(
    SOURCE_DOMAINS,
    source_val_sets
):
    if len(val_set) < EXAMPLES_PER_SOURCE:
        raise ValueError(
            f"{domain} validation set contains only "
            f"{len(val_set)} examples; "
            f"{EXAMPLES_PER_SOURCE} are required."
        )

    # Random sample without replacement, determined solely
    # by SHARPNESS_SEED.
    indices = torch.randperm(
        len(val_set),
        generator=sharpness_generator
    )[:EXAMPLES_PER_SOURCE]

    domain_images = []
    domain_labels = []

    for idx in indices.tolist():
        image, label = val_set[idx]

        domain_images.append(image)
        domain_labels.append(label)

    domain_images = torch.stack(domain_images)
    domain_labels = torch.tensor(
        domain_labels,
        dtype=torch.long
    )

    batch_images.append(domain_images)
    batch_labels.append(domain_labels)

    print(
        f"{domain:15s}: selected "
        f"{len(domain_labels)} validation examples"
    )


# 32 examples/domain x 3 domains = 96 examples
sharpness_images = torch.cat(
    batch_images,
    dim=0
).to(device)

sharpness_labels = torch.cat(
    batch_labels,
    dim=0
).to(device)


print(
    f"\nFixed sharpness batch size: "
    f"{sharpness_images.size(0)}"
)

assert sharpness_images.size(0) == (
    EXAMPLES_PER_SOURCE * len(SOURCE_DOMAINS)
)


# ============================================================
# 2. Local sharpness function
# ============================================================

def compute_local_sharpness(
    current_model,
    images,
    labels,
    rho=0.05
):
    """
    Compute

        Delta_sharp =
            L_val(theta + epsilon) - L_val(theta)

    where

        epsilon =
            rho * grad(L_val) / ||grad(L_val)||_2

    using a global L2 norm over all parameter gradients.

    The model stays in eval mode throughout the diagnostic.
    Parameters are restored exactly before returning.
    """

    # --------------------------------------------------------
    # Evaluation mode as required by the protocol.
    #
    # Gradients are still enabled; eval() only changes the
    # behavior of modules such as BatchNorm / Dropout.
    # --------------------------------------------------------

    current_model.eval()
    current_model.zero_grad(set_to_none=True)


    # ========================================================
    # Baseline validation loss L_val(theta)
    # ========================================================

    logits = current_model(images)

    base_loss = sharpness_criterion(
        logits,
        labels
    )

    # Need gradients with respect to model parameters.
    base_loss.backward()


    # ========================================================
    # Global gradient norm
    #
    # ||grad L||_2 =
    # sqrt(sum_i ||grad_i||_2^2)
    # ========================================================

    grad_sq_sum = torch.zeros(
        (),
        device=images.device
    )

    for parameter in current_model.parameters():

        if parameter.grad is not None:

            grad_sq_sum += (
                parameter.grad.detach()
                .pow(2)
                .sum()
            )

    grad_norm = torch.sqrt(grad_sq_sum)


    if grad_norm.item() == 0.0:
        raise RuntimeError(
            "Gradient norm is zero; "
            "the sharpness perturbation cannot be constructed."
        )


    # ========================================================
    # Perturb:
    #
    # epsilon =
    #     rho * grad(L_val) / ||grad(L_val)||_2
    #
    # Save each epsilon so theta can be restored exactly.
    # ========================================================

    perturbations = []

    with torch.no_grad():

        for parameter in current_model.parameters():

            if parameter.grad is None:
                perturbations.append(None)
                continue

            epsilon = (
                rho
                * parameter.grad
                / grad_norm
            )

            parameter.add_(epsilon)

            perturbations.append(
                epsilon.clone()
            )


    # ========================================================
    # Perturbed validation loss L_val(theta + epsilon)
    #
    # No backward pass is needed here because this diagnostic
    # uses exactly one gradient-ascent perturbation.
    # ========================================================

    with torch.no_grad():

        perturbed_logits = current_model(
            images
        )

        perturbed_loss = sharpness_criterion(
            perturbed_logits,
            labels
        )


    # ========================================================
    # Restore theta exactly
    # ========================================================

    with torch.no_grad():

        for parameter, epsilon in zip(
            current_model.parameters(),
            perturbations
        ):

            if epsilon is not None:
                parameter.sub_(epsilon)


    current_model.zero_grad(set_to_none=True)


    # ========================================================
    # Local sharpness proxy
    # ========================================================

    delta_sharp = (
        perturbed_loss.item()
        - base_loss.item()
    )


    return {
        "L_val(theta)": base_loss.item(),
        "grad_norm": grad_norm.item(),
        "L_val(theta+epsilon)": perturbed_loss.item(),
        "Delta_sharp": delta_sharp
    }


# ============================================================
# 3. Run EXACTLY the same diagnostic on all three models
# ============================================================

sharpness_results = {}

for method_name, current_model in models.items():

    result = compute_local_sharpness(
        current_model,
        sharpness_images,
        sharpness_labels,
        rho=SHARPNESS_RHO
    )

    sharpness_results[method_name] = result

    print(
        f"\n{method_name} — Local Sharpness"
    )
    print("=" * 60)

    print(
        f"L_val(theta)          : "
        f"{result['L_val(theta)']:.6f}"
    )

    print(
        f"||grad L_val||_2      : "
        f"{result['grad_norm']:.6f}"
    )

    print(
        f"L_val(theta+epsilon)  : "
        f"{result['L_val(theta+epsilon)']:.6f}"
    )

    print(
        f"Delta_sharp           : "
        f"{result['Delta_sharp']:.6f}"
    )


# ============================================================
# 4. Compact comparison table
# ============================================================

sharpness_df = pd.DataFrame.from_dict(
    sharpness_results,
    orient="index"
)

sharpness_df.index.name = "Method"

print("\n\nLocal Sharpness Comparison")
print("=" * 80)

display(
    sharpness_df.round(6)
)


# ------------------------------------------------------------
# Optional ordering:
# smaller Delta_sharp = locally less sensitive / flatter
# under this standardized diagnostic.
# ------------------------------------------------------------

print("\nModels ordered by Delta_sharp (smaller is flatter locally):")

display(
    sharpness_df[
        ["Delta_sharp"]
    ].sort_values("Delta_sharp").round(6)
)

photo          : selected 32 validation examples
art_painting   : selected 32 validation examples
cartoon        : selected 32 validation examples

Fixed sharpness batch size: 96

ERM — Local Sharpness
L_val(theta)          : 0.345641
||grad L_val||_2      : 7.602632
L_val(theta+epsilon)  : 0.796217
Delta_sharp           : 0.450576

DAN-DG — Local Sharpness
L_val(theta)          : 0.330760
||grad L_val||_2      : 3.444336
L_val(theta+epsilon)  : 0.512610
Delta_sharp           : 0.181849

SAM — Local Sharpness
L_val(theta)          : 0.199992
||grad L_val||_2      : 3.877777
L_val(theta+epsilon)  : 0.407216
Delta_sharp           : 0.207223


Local Sharpness Comparison


,L_val(theta),grad_norm,L_val(theta+epsilon),Delta_sharp
Method,,,,
ERM,0.345641,7.602632,0.796217,0.450576
DAN-DG,0.330760,3.444336,0.512610,0.181849
SAM,0.199992,3.877777,0.407216,0.207223



Models ordered by Delta_sharp (smaller is flatter locally):


,Delta_sharp
Method,
DAN-DG,0.181849
SAM,0.207223
ERM,0.450576


# Controlled Design Study

In [12]:
# ============================================================
# Controlled Design Study — SAM rho
# rho in {0.01, 0.05, 0.1}
#
# Expected before running:
# - rho=0.01: weaker SAM regularization; likely sharper solution.
# - rho=0.05: main/default SAM setting.
# - rho=0.10: stronger regularization; may reduce sharpness,
#   but too much may hurt source and Sketch performance.
#
# Main comparison remains rho=0.05 regardless of Sketch results.
# ============================================================

import copy
import pandas as pd

RHO_VALUES = [0.01, 0.05, 0.10]
study_results = []


def train_sam_with_rho(rho):

    # Reuse the already-trained main SAM model for rho=0.05
    if rho == 0.05:
        m = PACSResNet18(num_classes=7).to(device)
        m.load_state_dict(
            torch.load(
                "sam-rho-0.05.pt",
                map_location=device,
                weights_only=True
            )
        )
        return m

    # Same seed and all other settings as the main SAM experiment
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    m = PACSResNet18(num_classes=7).to(device)

    opt = torch.optim.AdamW(
        m.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    best_f1 = -1
    best_state = None

    for epoch in range(NUM_EPOCHS):

        print(f"rho={rho} | Epoch {epoch + 1}/{NUM_EPOCHS}")

        m.train()
        freeze_batchnorm_running_stats(m)

        for images, labels in train_loader_source:

            images = images.to(device)
            labels = labels.to(device)

            # SAM pass 1
            opt.zero_grad()
            logits = m(images)
            loss = criterion(logits, labels)
            loss.backward()

            grad_norm = sam_gradient_norm(m)
            scale = rho / (grad_norm + SAM_EPS)

            perturbations = {}

            with torch.no_grad():
                for p in m.parameters():
                    if p.grad is not None:
                        e = p.grad * scale
                        p.add_(e)
                        perturbations[p] = e

            # SAM pass 2
            opt.zero_grad()

            m.train()
            freeze_batchnorm_running_stats(m)

            loss_perturbed = criterion(
                m(images),
                labels
            )
            loss_perturbed.backward()

            # Restore original parameters
            with torch.no_grad():
                for p, e in perturbations.items():
                    p.sub_(e)

            opt.step()

        # Same source-validation model selection
        _, val_f1 = evaluate_sam(
            m,
            val_loader_source
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(m.state_dict())

    m.load_state_dict(best_state)
    return m


# ============================================================
# Run controlled study
# ============================================================

for rho in RHO_VALUES:

    print(f"\nRunning rho = {rho}")

    m = train_sam_with_rho(rho)

    # Source-domain performance
    source_accs = []
    source_f1s = []

    for domain in SOURCE_DOMAINS:

        acc, f1 = evaluate_sam(
            m,
            source_val_loaders[domain]
        )

        source_accs.append(acc)
        source_f1s.append(f1)

    # Sketch performance
    sketch_acc, sketch_f1 = evaluate_sam(
        m,
        target_loader
    )

    # SAM-specific diagnostic:
    # use the SAME fixed rho=0.05 diagnostic from the notebook
    sharp = compute_local_sharpness(
        m,
        sharpness_images,
        sharpness_labels,
        rho=SHARPNESS_RHO
    )

    study_results.append({
        "rho": rho,
        "Source Mean Acc": np.mean(source_accs),
        "Source Mean F1": np.mean(source_f1s),
        "Source Worst Acc": np.min(source_accs),
        "Source Worst F1": np.min(source_f1s),
        "Delta Sharp": sharp["Delta_sharp"],
        "Sketch Acc": sketch_acc,
        "Sketch F1": sketch_f1
    })

    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Results
# ============================================================

sam_rho_df = pd.DataFrame(study_results)

print("\nSAM rho Controlled Study")
print("Main/default setting remains rho = 0.05.")

display(
    sam_rho_df.round(4)
)


Running rho = 0.01
rho=0.01 | Epoch 1/30
rho=0.01 | Epoch 2/30
rho=0.01 | Epoch 3/30
rho=0.01 | Epoch 4/30
rho=0.01 | Epoch 5/30
rho=0.01 | Epoch 6/30
rho=0.01 | Epoch 7/30
rho=0.01 | Epoch 8/30
rho=0.01 | Epoch 9/30
rho=0.01 | Epoch 10/30
rho=0.01 | Epoch 11/30
rho=0.01 | Epoch 12/30
rho=0.01 | Epoch 13/30
rho=0.01 | Epoch 14/30
rho=0.01 | Epoch 15/30
rho=0.01 | Epoch 16/30
rho=0.01 | Epoch 17/30
rho=0.01 | Epoch 18/30
rho=0.01 | Epoch 19/30
rho=0.01 | Epoch 20/30
rho=0.01 | Epoch 21/30
rho=0.01 | Epoch 22/30
rho=0.01 | Epoch 23/30
rho=0.01 | Epoch 24/30
rho=0.01 | Epoch 25/30
rho=0.01 | Epoch 26/30
rho=0.01 | Epoch 27/30
rho=0.01 | Epoch 28/30
rho=0.01 | Epoch 29/30
rho=0.01 | Epoch 30/30

Running rho = 0.05

Running rho = 0.1
rho=0.1 | Epoch 1/30
rho=0.1 | Epoch 2/30
rho=0.1 | Epoch 3/30
rho=0.1 | Epoch 4/30
rho=0.1 | Epoch 5/30
rho=0.1 | Epoch 6/30
rho=0.1 | Epoch 7/30
rho=0.1 | Epoch 8/30
rho=0.1 | Epoch 9/30
rho=0.1 | Epoch 10/30
rho=0.1 | Epoch 11/30
rho=0.1 | Epoch 12/30
rho=0

,rho,Source Mean Acc,Source Mean F1,Source Worst Acc,Source Worst F1,Delta Sharp,Sketch Acc,Sketch F1
0,0.01,0.9548,0.9529,0.9268,0.9256,0.4079,0.6050,0.6076
1,0.05,0.9606,0.9620,0.9390,0.9421,0.2072,0.7157,0.7295
2,0.10,0.9443,0.9424,0.9122,0.9099,0.1793,0.6582,0.6577
